In [ ]:
import os
import sys

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

    colab_src_dir = "/content/drive/MyDrive/Luca/research/unsupervised-feature-research/src"
    os.chdir(colab_src_dir)
    if colab_src_dir not in sys.path:
        sys.path.insert(0, colab_src_dir)

### Setup

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import math

from models import Autoencoder, K_Sparse_Autoencoder, FC_WTA_Autoencoder
from datasets import get_data_loader, get_flattened_size, get_patch_shape

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device}")

In [ ]:
k              = 50
epochs         = 5000
batch_size     = 128
bottleneck_dim = 1000
lr             = 0.01
momentum       = 0.9     
dataset        = "cifar10_patches_color"  # or "mnist", "mnist_patches", "cifar10_patches"

### Dataset

In [ ]:
os.makedirs('../data', exist_ok=True)

In [ ]:
data_loader = get_data_loader(dataset, batch_size=batch_size)

### Autoencoders

In [ ]:
input_dim = get_flattened_size(dataset)

normal_autoencoder = Autoencoder(
                        dim=(input_dim, bottleneck_dim)
                    ).to(device)
normal_criterion   = nn.MSELoss()
normal_optimizer   = torch.optim.SGD(normal_autoencoder.parameters(), lr=lr, momentum=momentum)

sparse_autoencoder = K_Sparse_Autoencoder(
                        dim=(input_dim, bottleneck_dim), 
                        k=k, 
                        total_epochs=epochs,
                        dataset_size=len(data_loader.dataset)
                    ).to(device)
sparse_criterion   = nn.MSELoss()
sparse_optimizer   = torch.optim.SGD(sparse_autoencoder.parameters(), lr=lr, momentum=momentum)

fc_wta_autoencoder = FC_WTA_Autoencoder(
                        dim=(input_dim, bottleneck_dim), 
                        k=k
                    ).to(device)
fc_wta_criterion   = nn.MSELoss()
fc_wta_optimizer   = torch.optim.SGD(fc_wta_autoencoder.parameters(), lr=lr, momentum=momentum)

### Training

In [ ]:
normal_losses = []
sparse_losses = []
fc_wta_losses = []

normal_autoencoder.train()
sparse_autoencoder.train()
fc_wta_autoencoder.train()

epochbar  = tqdm(range(epochs), desc="Epochs")
n_batches = len(data_loader)
log_every = max(1, n_batches // 4)

for epoch in epochbar:
    normal_epoch_loss = 0.0
    sparse_epoch_loss = 0.0
    fc_wta_epoch_loss = 0.0
    inputs_processed_in_epoch = 0

    for step, (batch_inputs, _) in enumerate(data_loader):
        batch_size = batch_inputs.shape[0]
        batch_inputs = batch_inputs.to(device)
        normal_optimizer.zero_grad()
        sparse_optimizer.zero_grad()
        fc_wta_optimizer.zero_grad()

        normal_x_recon = normal_autoencoder(batch_inputs)
        sparse_x_recon = sparse_autoencoder(
            batch_inputs,
            epoch=epoch,
            inputs_processed_in_epoch=inputs_processed_in_epoch,
        )
        fc_wta_x_recon = fc_wta_autoencoder(batch_inputs)

        inputs_processed_in_epoch += batch_size

        normal_loss = normal_criterion(normal_x_recon, batch_inputs)
        sparse_loss = sparse_criterion(sparse_x_recon, batch_inputs)
        fc_wta_loss = fc_wta_criterion(fc_wta_x_recon, batch_inputs)

        normal_loss.backward()
        sparse_loss.backward()
        fc_wta_loss.backward()

        normal_optimizer.step()
        sparse_optimizer.step()
        fc_wta_optimizer.step()

        normal_batch_loss = normal_loss.item()
        sparse_batch_loss = sparse_loss.item()
        fc_wta_batch_loss = fc_wta_loss.item()

        normal_epoch_loss += normal_batch_loss
        sparse_epoch_loss += sparse_batch_loss
        fc_wta_epoch_loss += fc_wta_batch_loss

        if (step + 1) % log_every == 0:
            avg_normal = normal_epoch_loss / (step + 1)
            avg_sparse = sparse_epoch_loss / (step + 1)
            avg_fc_wta = fc_wta_epoch_loss / (step + 1)
            normal_losses.append(avg_normal)
            sparse_losses.append(avg_sparse)
            fc_wta_losses.append(avg_fc_wta)

        epochbar.set_postfix({
            "normal_loss": f"{normal_batch_loss:.4f}",
            "sparse_loss": f"{sparse_batch_loss:.4f}",
            "sparse_k":    f"{sparse_autoencoder.last_k:.2f}",
            "fc_wta_loss": f"{fc_wta_batch_loss:.4f}"
        })

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

xs = [i / 4 for i in range(1, len(normal_losses) + 1)]

axes[0].plot(xs, normal_losses, "o-")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mean Train Loss")
axes[0].set_title("Normal Autoencoder Training Performance")
axes[0].grid(True, alpha=0.3)

axes[1].plot(xs, sparse_losses, "o-", color='orange')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean Train Loss")
axes[1].set_title("Sparse Autoencoder Training Performance")
axes[1].grid(True, alpha=0.3)

axes[2].plot(xs, fc_wta_losses, "o-", color='green')
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Mean Train Loss")
axes[2].set_title("FC-WTA Autoencoder Training Performance")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Visualizations

In [ ]:
patches_batch, _ = next(iter(data_loader))
n_show = min(25, patches_batch.size(0))
try:
    C, H, W = get_patch_shape(dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s
patches_vis = patches_batch[:n_show].reshape(-1, C, H, W)
grid_patches = make_grid(patches_vis, nrow=5, normalize=True, scale_each=True, padding=2)
plt.figure(figsize=(8, 8))
plt.imshow(grid_patches.permute(1, 2, 0).clamp(0, 1))
plt.axis("off")
plt.title(f"Sample {dataset} patches (n={n_show})")
plt.tight_layout()
plt.show()

In [ ]:
def maximal_activation_normalize(W):
    return W / torch.sqrt(torch.sum(W**2, dim=1, keepdim=True) + 1e-8)

# Patch shape (C, H, W) for grayscale or color
try:
    C, H, W = get_patch_shape(dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s

W_enc_normal = maximal_activation_normalize(normal_autoencoder.encoder.weight.detach().cpu())
W_enc_sparse = maximal_activation_normalize(sparse_autoencoder.encoder.weight.detach().cpu())
W_enc_fc_wta = maximal_activation_normalize(fc_wta_autoencoder.encoder.weight.detach().cpu())

max_filters = 100
kernels_normal = W_enc_normal[:max_filters].reshape(-1, C, H, W)
kernels_sparse = W_enc_sparse[:max_filters].reshape(-1, C, H, W)
kernels_fc_wta = W_enc_fc_wta[:max_filters].reshape(-1, C, H, W)

grid_normal = make_grid(kernels_normal, nrow=10, normalize=True, scale_each=True, padding=1)
grid_sparse = make_grid(kernels_sparse, nrow=10, normalize=True, scale_each=True, padding=1)
grid_fc_wta = make_grid(kernels_fc_wta, nrow=10, normalize=True, scale_each=True, padding=1)

fig = plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
plt.imshow(grid_normal.permute(1, 2, 0).clamp(0, 1))
plt.axis('off')
plt.title('Normal Autoencoder Weights')

plt.subplot(1, 3, 2)
plt.imshow(grid_sparse.permute(1, 2, 0).clamp(0, 1))
plt.axis('off')
plt.title('Sparse Autoencoder Weights')

plt.subplot(1, 3, 3)
plt.imshow(grid_fc_wta.permute(1, 2, 0).clamp(0, 1))
plt.axis('off')
plt.title('FC-WTA Autoencoder Weights')

plt.tight_layout()
plt.show()

In [ ]:
# Use patch shape from previous cell (C, H, W)
try:
    C, H, W = get_patch_shape(dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s

normal_autoencoder.eval()
sparse_autoencoder.eval()
fc_wta_autoencoder.eval()

batch_inputs, _ = next(iter(data_loader))
batch_inputs = batch_inputs.to(device)

with torch.no_grad():
    act_normal = torch.sigmoid(normal_autoencoder.encoder(batch_inputs))

    act_sparse_raw = sparse_autoencoder.encoder(batch_inputs)
    k_eval = min(max(1, sparse_autoencoder.k), sparse_autoencoder.bottleneck_dim)
    _, sparse_topk_idx = torch.topk(act_sparse_raw, k_eval, dim=1)
    sparse_mask = torch.zeros_like(act_sparse_raw).scatter_(1, sparse_topk_idx, 1)
    act_sparse = act_sparse_raw * sparse_mask

    act_fc_wta = fc_wta_autoencoder.relu(fc_wta_autoencoder.encoder(batch_inputs))

sample_idx = torch.randint(0, batch_inputs.shape[0], (1,)).item()

W_enc_normal = normal_autoencoder.encoder.weight.detach().cpu()
W_enc_sparse = sparse_autoencoder.encoder.weight.detach().cpu()
W_enc_fc_wta = fc_wta_autoencoder.encoder.weight.detach().cpu()

a_normal = act_normal[sample_idx].cpu().unsqueeze(1)  # (bottleneck_dim, 1)
a_sparse = act_sparse[sample_idx].cpu().unsqueeze(1)
a_fc_wta = act_fc_wta[sample_idx].cpu().unsqueeze(1)

max_filters = 100
act_kernels_normal = (a_normal * W_enc_normal)[:max_filters].reshape(-1, C, H, W)
act_kernels_sparse = (a_sparse * W_enc_sparse)[:max_filters].reshape(-1, C, H, W)
act_kernels_fc_wta = (a_fc_wta * W_enc_fc_wta)[:max_filters].reshape(-1, C, H, W)

grid_act_normal = make_grid(act_kernels_normal, nrow=10, normalize=True, scale_each=True, padding=1)
grid_act_sparse = make_grid(act_kernels_sparse, nrow=10, normalize=True, scale_each=True, padding=1)
grid_act_fc_wta = make_grid(act_kernels_fc_wta, nrow=10, normalize=True, scale_each=True, padding=1)

fig = plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
plt.imshow(grid_act_normal.permute(1, 2, 0).clamp(0, 1))
plt.axis('off')
plt.title(f'Normal AE Activations (sample {sample_idx})')

plt.subplot(1, 3, 2)
plt.imshow(grid_act_sparse.permute(1, 2, 0).clamp(0, 1))
plt.axis('off')
plt.title(f'K-Sparse AE Activations (sample {sample_idx})')

plt.subplot(1, 3, 3)
plt.imshow(grid_act_fc_wta.permute(1, 2, 0).clamp(0, 1))
plt.axis('off')
plt.title(f'FC-WTA AE Activations (sample {sample_idx})')

plt.tight_layout()
plt.show()